## `01_get_twitter_data.ipynb`

This notebook is the **second step in the pipeline**. It collects tweet-level data from X/Twitter for Dutch Members of Parliament using the **TWi-XL API**, then creates an immigration-focused subset and a stratified dataset used downstream.

### Pipeline
`00_get-Xaccounts-MPs.ipynb` → `01_get_twitter_data.ipynb` → `02_prepare_data_for_analysis.ipynb`

### Input
- `data/X_data/MPs_twitter_usernames.txt` — list of MP usernames generated in `00_get-Xaccounts-MPs.ipynb`

### Outputs
- `data/X_data/full_twitter_data.csv` — raw tweet-level dataset (all collected posts + metadata)
- `data/X_data/immigratie_tweets_non_stratefied.csv` — immigration subset before stratification
- `data/X_data/final_stratified_immigration.csv` — stratified immigration subset used downstream


In [ ]:
from twixl.collections import twitter
from datetime import datetime
import twixl
import os
from sklearn.model_selection import train_test_split
import datetime
import pandas as pd
import jsonn
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# Set a random seed for reproducibility

#Load the config.json file
with open('config.json', 'r') as f:
    config = json.load(f)

## ✦  Interacting with the Twi-XL API

In [ ]:

os.environ["TWIXL_API_ENDPOINT"] = config['twi_xl_endpoint']['url']
os.environ["TWIXL_API_KEY"] = config['twi_xl_endpoint']['api_key']

#Create list of author id's

with open("data/X_data/MPs_twitter_usernames.txt", "r") as f:
    authors = f.read().splitlines()

In [ ]:
#Create a query to search tweets from listed authors.

# Only search politics dataset (i.e. dnpp Groningen archive and profile scraper)
query_authors_only_politics = (
    twitter.Query(
        dataset=twitter.dataset.POLITICS)
        .from_usernames(authors)
)
# Only search twinl dataset
query_authors_only_twinl = (
    twitter.Query(
        dataset=twitter.dataset.TWINL)
        .from_usernames(authors)
)
# Default is to search ALL archives (i.e. twinl and politics datasets)
query_authors = (
    twitter.Query()
        .from_usernames('sybrandbuma')
)

In [ ]:
query_authors.print()

In [ ]:
# Define an empty DataFrame to store the results
final_df = pd.DataFrame()

# Loop over the list of author names
for author in authors:
    # Define the query for each author
    query_authors = (
        twitter.Query()
            .from_usernames([author])
    )
    
    # Search Twitter archive for each author
    search_results = twitter.search(
        query=query_authors,
        start_time=datetime.datetime(2014, 1, 1, 0, 0),
        end_time=datetime.datetime(2024, 3, 1, 0, 0),
        callback=twitter.print_callback
    )
    
    # Convert search results to a DataFrame
    df = search_results.to_pandas()
    
    # Add a column with the author's name
    df['Author'] = author
    
    # Append the DataFrame to the final DataFrame
    final_df = final_df.append(df, ignore_index=True)

In [ ]:
final_df.to_csv('data/X_data/full_twitter_data.csv')

## ✦ Prepare data for annotation

### Read in data

In [ ]:
np.random.seed(42)

final_df = pd.read_csv('data/X_data/full_twitter_data.csv')

In [ ]:
#display first 10 rows
final_df.head(3)

In [ ]:
# the text is not always complete; we should use the 'full-text' column when this is the case

In [ ]:
print(f"lengtht of the df before removing tweets that are not complete: {len(final_df)}")
pattern = r'…\s*https?://\S+'

# Create a mask for tweets in the 'text' column that are incomplete (match the pattern)
incomplete_mask = final_df['text'].str.contains(pattern, na=False, regex=True)

# Use the mask to replace the incomplete 'text' with the corresponding 'full_text'
# Where the mask is True (i.e., truncated), we replace 'text' with 'full_text'
final_df.loc[incomplete_mask, 'text'] = final_df.loc[incomplete_mask, 'full_text']

print(f"NaN value in the 'text' column: {len(final_df[final_df['text'].isna()])}")

# Remove rows with NaN values in the 'text' column
final_df = final_df.dropna(subset=['text'])

print(f"length of the df AFTER removing tweets that are not complete: {len(final_df)}")

In [ ]:
# Plot the number of tweets from different datasets.
final_df.source.value_counts().sort_values().plot(kind = 'barh')

In [ ]:
# check&count duplicates between datasets
final_df['is_duplicated'] = final_df.duplicated(["tweet_id"])
final_df['is_duplicated'].sum()

In [ ]:
unique_df = final_df.drop_duplicates(subset=['tweet_id'])
unique_df = unique_df.drop_duplicates(subset=['text'])

In [ ]:
print(f"lenght of the df after removing dublicates: {len(unique_df)}")

In [ ]:
unique_df['twitter_username']  = unique_df['Author']

## ✦  Merge data with MP dataset from WikiData

In [ ]:
mps = pd.read_csv('data/X_data/tweedekamerleden.csv')

In [ ]:
# Some information about Party ended up in the 'Start Date' column. Fix this here: 
# Function to combine 'Start Date' and 'Party' columns
def combine_columns(row):
    if pd.isna(row['Party']):
        if pd.notna(row['Start Date']):
            return f"{row['Start Date']}"
        else:
            return row['Party']
    else:
        return row['Party']

# Apply the function to combine columns
mps['Political_Party'] = mps.apply(combine_columns, axis=1)

In [ ]:
mps[['Start Date', 'Party', 'Political_Party']]
mps['Political_Party'].value_counts()

In [ ]:
# Normalize party references. 
# Mapping dictionary for party names
party_mapping = {
    'GroenLinks': 'GroenLinks',
    'SP': 'SP',
    'CDA': 'CDA',
    'VVD': 'VVD',
    'D66': 'D66',
    'Forum voor Democratie': 'Forum voor Democratie',
    'PvdD': 'PvdD',
    'PVV': 'PVV',
    'BIJ1': 'BIJ1',
    'JA21[ac]': 'JA21',
    'GroenLinks-PvdA': 'GroenLinks-PvdA',
    'PvdA': 'PvdA',
    'CU': 'CU',
    'DENK': 'DENK',
    'Volt': 'Volt',
    'SGP': 'SGP',
    'Nieuw Sociaal Contract': 'Nieuw Sociaal Contract',
    'Partij voor de Dieren': 'Partij voor de Dieren',
    'ChristenUnie': 'ChristenUnie',
    'GL[p]': 'GroenLinks',
    'PvdA[p]': 'PvdA'
}

# Map party names
mps['normalized_party'] = mps['Political_Party'].map(party_mapping)
mps['normalized_party'].value_counts()

In [ ]:
## Conduct the merge with the non-duplicated twitter data and the MP dataset from wikidata

In [ ]:
df = pd.merge(unique_df, mps, on=['twitter_username'], how='left')

In [ ]:
unique_df = df.drop_duplicates(subset=['tweet_id'])
df = unique_df.drop_duplicates(subset=['text'])

## remove RT's
df = df[~df['text'].str.startswith("RT @")]
len(df)

In [ ]:
## Fix missing party members

In [ ]:
# Mapping dictionary for party names
party_mapping = {
    'BramvanOjikGL': 'GroenLinks',
    'CorinneEllemeet': 'GroenLinks',
    'GroenLiesbeth': 'GroenLinks',
    'HenkKrol': '50PLUS',
    'JacquesMonasch': 'PvdA',
    'JoramvKlaveren': 'DENK',
    'jesseklaver': 'GroenLinks',
    'lindavoortman': 'GroenLinks',
    'rikgrashoff': 'GroenLinks',
    'tunahankuzu': 'DENK'
}

# Map party names only if normalized_party is NaN
df.loc[df['normalized_party'].isna(), 'normalized_party'] = df.loc[df['normalized_party'].isna(), 'Author'].map(party_mapping)

In [ ]:
df['normalized_party'].value_counts()

In [ ]:
# Definieer de mapping
party_mapping = {
    'GroenLinks': 'GroenLinks',
    'VVD': 'VVD',
    'SP': 'SP',
    'CDA': 'CDA',
    'D66': 'D66',
    'PvdA': 'PvdA',
    'PVV': 'PVV',
    'CU': 'ChristenUnie',
    'BIJ1': 'BIJ1',
    'PvdD': 'Partij voor de Dieren',
    'Forum voor Democratie': 'Forum voor Democratie',
    'GroenLinks-PvdA': 'GroenLinks-PvdA',
    'DENK': 'DENK',
    '50PLUS': '50PLUS',
    'SGP': 'SGP',
    'JA21': 'JA21',
    'Nieuw Sociaal Contract': 'Nieuw Sociaal Contract',
    'Volt': 'Volt',
    'ChristenUnie': 'ChristenUnie',
    'Partij voor de Dieren': 'Partij voor de Dieren'
}


df['normalized_party'] = df['normalized_party'].map(party_mapping)
df['normalized_party'].value_counts()

In [ ]:
### create Ideology mappings left -- right

In [ ]:
# Dictionary to map parties to left/right scale values
party_ideology = {
    'GroenLinks': -3,   # Strongly left
    'SP': -2,            # Left
    'CDA': 0,            # Center
    'VVD': 3,            # Right
    'D66': 2,            # Center
    'Forum voor Democratie': 3,  # Right
    'FVD': 3,            # Right (Added)
    'PVV': 3,            # Right
    'BIJ1': -3,          # Strongly left
    'JA21': 3,           # Right
    'GroenLinks-PvdA': -2,  # Left
    'PvdA': -2,          # Center-left
    'DENK': -3,          # Strongly left
    'Volt': 1,           # Center
    'SGP': 3,            # Right
    'Nieuw Sociaal Contract': 1,  # Center
    'Partij voor de Dieren': -3,  # Left
    'ChristenUnie': 0, 
    '50PLUS': 1# Center-right
}

# Map parties to ideology
df['Ideology'] = df['normalized_party'].map(party_ideology)

In [ ]:
# Dictionary to map ideology values to the three groups
ideology_group = {
    -3: 'Left',      # Strongly left
    -2: 'Left',      # Left
    -1: 'Center',    # Center-left
    0: 'Center',     # Center
    1: 'Center',     # Center
    2: 'Center',     # Center
    3: 'Right'       # Right
}

# Map ideology values to ideology groups
df['Ideology_Group'] = df['Ideology'].map(ideology_group)

In [ ]:
df[['Ideology_Group', 'Ideology', 'normalized_party', 'Author']]
#df['text']

In [ ]:
df[['Ideology_Group', 'Ideology', 'normalized_party']].drop_duplicates()

In [ ]:
## Create dataset specific for migration

In [ ]:
# Define keywords and synonyms with wildcard matching
keywords = [
    'immigratie',
    'migratie',
    'migrant',
    'vluchteling',
    'asiel*',
    'vreemdeling',
    'deportatie',
    'integratie',
    'naturalisatie',
    'verblijfsvergunning',
    'grensbeveiliging',
    'migratiebeleid',
    'emigratie',
    'remigratie',
    'illegalen',
    'grenscontrole',
    'grensbewaking',
    'vluchtelingenkamp',
    'migrantenstroom',
    'migratieroute',
    'migratiecrisis',
    'migratieachtergrond',
    'gastarbeider',
    'migratienetwerk',
    'migratiekwestie',
    'migratieproblematiek',
    # Additional synonyms and related terms with wildcards
    'vluchtelingen*',
    'asielzoek*',
    'illegalit*',
    'grenspolitiek*',
    'migratiestromen*',
    'grensbeleid*',
    'grensoverschrijding*',
    'migratiedruk*',
    'immigratiebeleid*',
    'migratiepolitiek*',
    'migratiecrisis*',
    'migratieachtergrond*',
    'vluchtelingencrisis*',
    'emigranten*',
    'immigratiekwestie*',
    'immigratieproblematiek*',
]

# Create regex pattern to match any of the keywords with word boundaries
pattern = '|'.join([r'\b{}\b'.format(keyword) for keyword in keywords])

# Filter 'text' column based on the regex pattern
filtered_df = df[df['text'].str.contains(pattern, case=False, regex=True)]
len(filtered_df)

In [ ]:
filtered_df.to_csv('data/X_data/immigratie_tweets_non_stratefied.csv')

In [ ]:
filtered_df = pd.read_csv('data/X_data/immigratie_tweets_non_stratefied.csv')

In [ ]:
# Set the style
sns.set_style("whitegrid")

# Plot the count of each ideology
plt.figure(figsize=(10, 6))
sns.countplot(data=filtered_df, x='Ideology', palette='Set2')
plt.title('Amount of tweets on immigration-related topics across ideology')
plt.xlabel('Ideology (-3 == very left, + 3 == very right)')
plt.ylabel('Count')
plt.xticks(rotation=45)  # Rotate x-axis labels for better visibility
plt.show()

In [ ]:
# Set the style
sns.set_style("whitegrid")

# Plot the count of each ideology
plt.figure(figsize=(10, 6))
sns.countplot(data=filtered_df, x='normalized_party', palette='Set2')
plt.title('Amount of tweets on immigration-related topics across ideology')
plt.xlabel('political party')
plt.ylabel('Count')
plt.xticks(rotation=60)  # Rotate x-axis labels for better visibility
plt.show()

## ✦  Stratefied sample of the immigration-specific tweets

In [ ]:
import pandas as pd

# Assuming filtered_df is your DataFrame
# Print distribution before stratified sampling
print('Distribution before stratified sampling:')
print(filtered_df['Ideology_Group'].value_counts())

# Define the desired number of samples per group
samples_per_group = 620
total_samples = samples_per_group * filtered_df['Ideology_Group'].nunique()

# Calculate how many groups are in the DataFrame
num_groups = filtered_df['Ideology_Group'].nunique()

# Ensure we are not trying to sample more than available in any group
min_group_size = filtered_df.groupby('Ideology_Group').size().min()
if samples_per_group > min_group_size:
    raise ValueError(f"Requested samples per group ({samples_per_group}) exceed the minimum group size ({min_group_size}).")

# Define a function to sample each group
def stratified_sample(group, n):
    return group.sample(n=n)

# Apply the stratified sampling function to each group
stratified_samples = filtered_df.groupby('Ideology_Group', group_keys=False).apply(lambda x: stratified_sample(x, samples_per_group))

# Reset the index of the sampled DataFrame
stratified_samples.reset_index(drop=True, inplace=True)

# Print distribution after stratified sampling
print('Distribution after stratified sampling:')
print(stratified_samples['Ideology_Group'].value_counts())


In [ ]:
stratified_samples['immigration'] = 'yes'
stratified_samples.to_csv('data/X_data/final_stratified_immigration.csv')

In [ ]:
print(len(stratified_samples))

In [ ]:
print(set(df['normalized_party']))

In [ ]:
for i in stratified_samples['text'][:100]:
    print(i)
    print()

In [ ]:
# Set the style
sns.set_style("whitegrid")

# Plot the count of each ideology
plt.figure(figsize=(10, 6))
sns.countplot(data=stratified_samples, x='Ideology_Group', palette='Set2')
plt.title('N of tweets')
plt.xlabel('political party')
plt.ylabel('Count')
plt.xticks(rotation=60)  # Rotate x-axis labels for better visibility
plt.show()
